# Data Processing - CDC Chronic Disease Indicators

Downloading and cleaning the CDC dataset so we all have a clean state-level table to work from.

Data source: https://catalog.data.gov/dataset/u-s-chronic-disease-indicators

Team topics:
- Jonathan: Cardiovascular Disease + Insurance/Social Determinants
- Min Yu: Diabetes + Obesity
- Anuhya: Mental Health + Alcoholism
- Tsion: Cancer + COPD

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/cdc_chronic_disease_indicators.csv')
print("shape:", df.shape)
print("\ncolumns:")
print(list(df.columns))
df.head(3)

In [ ]:
# check what topics are in the dataset
print("unique topics (" + str(df['Topic'].nunique()) + "):\n")
for t in sorted(df['Topic'].unique()):
    count = len(df[df['Topic'] == t])
    print("  " + t + " (" + str(count) + " rows)")

In [ ]:
# checking what years and stratifications exist
print("years:", sorted(df['YearStart'].unique()))
print("\ndata value types:", df['DataValueType'].unique())
print("\nstratification categories:", df['StratificationCategory1'].unique())
print("\nstratification values (first 20):", df['Stratification1'].unique()[:20])

## Filtering to the indicators we need

We picked one or two key questions for each person's diseases, plus some shared risk factors like smoking and obesity. Using "Overall" stratification so we get the full population, and grabbing the most recent year that has good coverage.

In [ ]:
# each tuple is (question text from CDC, data value type, our short column name)
indicators = [
    # cardiovascular disease (jonathan)
    ("Diseases of the heart mortality among all people, underlying cause", "Age-adjusted Rate", "heart_disease_mortality_rate"),
    ("Coronary heart disease mortality among all people, underlying cause", "Age-adjusted Rate", "coronary_heart_mortality_rate"),
    ("Cerebrovascular disease (stroke) mortality among all people, underlying cause", "Age-adjusted Rate", "stroke_mortality_rate"),
    ("High blood pressure among adults", "Age-adjusted Prevalence", "high_blood_pressure_prev"),
    ("High cholesterol among adults who have been screened", "Age-adjusted Prevalence", "high_cholesterol_prev"),

    # social determinants (jonathan)
    ("Lack of health insurance among adults aged 18-64 years", "Crude Prevalence", "uninsured_prev"),
    ("Living below 150% of the poverty threshold among all people", "Crude Prevalence", "poverty_prev"),
    ("High school completion among adults aged 18-24", "Crude Prevalence", "hs_completion_prev"),
    ("Food insecure in the past 12 months among households", "Crude Prevalence", "food_insecurity_prev"),
    ("Lack of social and emotional support needed among adults", "Crude Prevalence", "lack_social_support_prev"),

    # diabetes (min yu)
    ("Diabetes among adults", "Age-adjusted Prevalence", "diabetes_prev"),
    ("Diabetes mortality among all people, underlying or contributing cause", "Age-adjusted Rate", "diabetes_mortality_rate"),

    # obesity / nutrition (min yu)
    ("Obesity among adults", "Age-adjusted Prevalence", "obesity_prev"),
    ("No leisure-time physical activity among adults", "Age-adjusted Prevalence", "physical_inactivity_prev"),
    ("Consumed fruit less than one time daily among adults", "Age-adjusted Prevalence", "low_fruit_prev"),
    ("Consumed vegetables less than one time daily among adults", "Age-adjusted Prevalence", "low_veggie_prev"),

    # mental health (anuhya)
    ("Depression among adults", "Age-adjusted Prevalence", "depression_prev"),
    ("Frequent mental distress among adults", "Age-adjusted Prevalence", "mental_distress_prev"),

    # alcohol (anuhya)
    ("Binge drinking prevalence among adults", "Age-adjusted Prevalence", "binge_drinking_prev"),
    ("Chronic liver disease mortality among all people, underlying cause", "Age-adjusted Rate", "liver_disease_mortality_rate"),

    # cancer (tsion)
    ("Invasive cancer (all sites combined), incidence", "Age-adjusted Rate", "cancer_incidence_rate"),
    ("Invasive cancer (all sites combined) mortality among all people, underlying cause", "Age-adjusted Rate", "cancer_mortality_rate"),
    ("Lung and bronchial cancer mortality among all people, underlying cause", "Age-adjusted Rate", "lung_cancer_mortality_rate"),

    # copd (tsion)
    ("Chronic obstructive pulmonary disease among adults", "Age-adjusted Prevalence", "copd_prev"),
    ("Chronic obstructive pulmonary disease mortality among adults aged 45 years and older, underlying cause", "Age-adjusted Rate", "copd_mortality_rate"),

    # shared risk factors
    ("Current cigarette smoking among adults", "Age-adjusted Prevalence", "smoking_prev"),
    ("Influenza vaccination among adults", "Age-adjusted Prevalence", "flu_vaccination_prev"),
]

print("selected", len(indicators), "indicators")

In [ ]:
# filter to overall stratification and just the 50 states + DC
# (no territories, no US aggregate)
US_STATES_AND_DC = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA',
    'KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
    'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT',
    'VA','WA','WV','WI','WY','DC'
]

filtered = df[
    (df['Stratification1'] == 'Overall') &
    (df['LocationAbbr'].isin(US_STATES_AND_DC))
].copy()

print("after filtering:", filtered.shape)
print("locations:", sorted(filtered['LocationAbbr'].unique()))
print("count:", filtered['LocationAbbr'].nunique())

In [ ]:
# for each indicator, grab the most recent year that has good state coverage
rows = []

for question, dvtype, short_name in indicators:
    sub = filtered[(filtered['Question'] == question) & (filtered['DataValueType'] == dvtype)]

    if sub.empty:
        print("WARNING: no data for " + short_name)
        continue

    # pick year with the most states reporting
    year_coverage = sub.groupby('YearStart')['LocationAbbr'].nunique().sort_index(ascending=False)
    best_year = year_coverage.idxmax()

    year_data = sub[sub['YearStart'] == best_year][['LocationAbbr', 'LocationDesc', 'DataValue']].copy()
    year_data['DataValue'] = pd.to_numeric(year_data['DataValue'], errors='coerce')
    year_data = year_data.rename(columns={'DataValue': short_name})
    year_data = year_data[['LocationAbbr', short_name]]

    n_states = year_data['LocationAbbr'].nunique()
    n_valid = year_data[short_name].notna().sum()
    print(short_name + ": year=" + str(best_year) + ", states=" + str(n_states) + ", valid=" + str(n_valid))

    rows.append(year_data)

## Building the state-level table

Now we merge everything together so each row is one state and each column is an indicator.

In [ ]:
# merge all the indicator dataframes together on state abbreviation
from functools import reduce

state_df = reduce(
    lambda left, right: pd.merge(left, right, on='LocationAbbr', how='outer'),
    rows
)

# add back the full state names
state_names = df[['LocationAbbr', 'LocationDesc']].drop_duplicates()
state_df = state_df.merge(state_names, on='LocationAbbr', how='left')

# put state info columns first
info_cols = ['LocationAbbr', 'LocationDesc']
indicator_cols = [c for c in state_df.columns if c not in info_cols]
state_df = state_df[info_cols + indicator_cols]

print("state table shape:", state_df.shape)
print("\nmissing values per column:")
print(state_df.isnull().sum())
state_df.head()

In [ ]:
# save to csv
state_df.to_csv('../data/processed/state_indicators.csv', index=False)
print("saved to data/processed/state_indicators.csv")
print("  " + str(state_df.shape[0]) + " states, " + str(state_df.shape[1] - 2) + " indicators")

print("\nsummary statistics:")
state_df.describe().round(1)